# Cross-Domain Benchmark

Cell2Cell에서 학습한 Full model과 Portable model을 BankChurners 외부 데이터에 각각 적용해 범용성을 비교하는 노트북이다.

- Full model: Cell2Cell 원본 56개 피처 기반
- Portable model: Cell2Cell과 BankChurners에서 공통 의미로 맞출 수 있는 15개 피처 기반
- 외부 평가는 BankChurners 전체를 테스트 세트로 사용한다.


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

ROOT = Path('/home/ms990728/소정해')
RESULT_PATH = ROOT / 'results' / 'cross_domain_benchmark_results.md'
RANDOM_STATE = 42
BEST_PARAMS = {
    'colsample_bytree': 0.8334271527847554,
    'gamma': 0.11429345433755263,
    'learning_rate': 0.06857996256539675,
    'max_depth': 3,
    'min_child_weight': 2,
    'n_estimators': 493,
    'reg_alpha': 0.2497327922401265,
    'reg_lambda': 0.9246782213565523,
    'subsample': 0.7954562418017752,
}
CREDIT_RATING_LABELS = ['7-Lowest', '6-VeryLow', '5-Low', '4-Medium', '3-Good', '2-High', '1-Highest']


def map_income_category(series: pd.Series) -> pd.Series:
    mapping = {
        'Less than $40K': 1,
        '$40K - $60K': 3,
        '$60K - $80K': 5,
        '$80K - $120K': 7,
        '$120K +': 9,
    }
    return series.map(mapping)


def map_marital_status(series: pd.Series) -> pd.Series:
    mapping = {
        'Married': 'Yes',
        'Single': 'No',
        'Divorced': 'No',
        'Unknown': 'Unknown',
        'Yes': 'Yes',
        'No': 'No',
    }
    return series.map(mapping)


def derive_credit_rating_bank(df: pd.DataFrame) -> pd.Series:
    score = df['Credit_Limit'].fillna(df['Credit_Limit'].median()) * (1 - df['Avg_Utilization_Ratio'].fillna(df['Avg_Utilization_Ratio'].median()))
    ranked = score.rank(method='first')
    return pd.qcut(ranked, 7, labels=CREDIT_RATING_LABELS)


def load_cell2cell(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=['NA', ''], keep_default_na=True)
    df['HandsetPrice'] = pd.to_numeric(df['HandsetPrice'].replace('Unknown', pd.NA), errors='coerce')
    df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1}).astype('Int64')
    return df


def load_bank(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['Attrition_Flag'] = df['Attrition_Flag'].map({'Existing Customer': 0, 'Attrited Customer': 1}).astype(int)
    return df


def build_full_cell_frame(df: pd.DataFrame):
    feature_cols = [c for c in df.columns if c not in {'CustomerID', 'Churn'}]
    X = df[feature_cols].copy()
    y = df['Churn'].astype(int)
    return X, y


def build_portable_cell_frame(df: pd.DataFrame):
    X = pd.DataFrame(index=df.index)
    X['tenure'] = df['MonthsInService']
    X['age'] = df['AgeHH1']
    X['income_group'] = df['IncomeGroup']
    X['marital_status'] = df['MaritalStatus']
    X['children_present'] = df['ChildrenInHH']
    X['relationship_depth'] = df['UniqueSubs'].fillna(0) + df['ActiveSubs'].fillna(0)
    X['transaction_volume'] = df['MonthlyMinutes']
    X['monetary_volume'] = df['MonthlyRevenue']
    X['balance_capacity'] = df['TotalRecurringCharge']
    X['utilization_or_overage'] = df['OverageMinutes']
    X['change_volume'] = df['PercChangeMinutes']
    X['change_amount'] = df['PercChangeRevenues']
    X['contact_intensity'] = df['RetentionCalls'].fillna(0) + df['CustomerCareCalls'].fillna(0) + df['MadeCallToRetentionTeam'].map({'Yes': 1, 'No': 0})
    X['credit_rating'] = df['CreditRating']
    X['credit_card_holder'] = df['HasCreditCard']
    y = df['Churn'].astype(int)
    return X, y


def build_full_bank_frame(df: pd.DataFrame):
    cell_cols = [
        'MonthlyRevenue', 'MonthlyMinutes', 'TotalRecurringCharge', 'DirectorAssistedCalls', 'OverageMinutes', 'RoamingCalls',
        'PercChangeMinutes', 'PercChangeRevenues', 'DroppedCalls', 'BlockedCalls', 'UnansweredCalls', 'CustomerCareCalls',
        'ThreewayCalls', 'ReceivedCalls', 'OutboundCalls', 'InboundCalls', 'PeakCallsInOut', 'OffPeakCallsInOut',
        'DroppedBlockedCalls', 'CallForwardingCalls', 'CallWaitingCalls', 'MonthsInService', 'UniqueSubs', 'ActiveSubs',
        'ServiceArea', 'Handsets', 'HandsetModels', 'CurrentEquipmentDays', 'AgeHH1', 'AgeHH2', 'ChildrenInHH',
        'HandsetRefurbished', 'HandsetWebCapable', 'TruckOwner', 'RVOwner', 'Homeownership', 'BuysViaMailOrder',
        'RespondsToMailOffers', 'OptOutMailings', 'NonUSTravel', 'OwnsComputer', 'HasCreditCard', 'RetentionCalls',
        'RetentionOffersAccepted', 'NewCellphoneUser', 'NotNewCellphoneUser', 'ReferralsMadeBySubscriber', 'IncomeGroup',
        'OwnsMotorcycle', 'AdjustmentsToCreditRating', 'HandsetPrice', 'MadeCallToRetentionTeam', 'CreditRating',
        'PrizmCode', 'Occupation', 'MaritalStatus'
    ]
    X = pd.DataFrame(index=df.index)
    for col in cell_cols:
        X[col] = np.nan

    X['MonthlyRevenue'] = df['Total_Trans_Amt'].astype(float)
    X['MonthlyMinutes'] = df['Total_Trans_Ct'].astype(float)
    X['TotalRecurringCharge'] = df['Credit_Limit'].astype(float)
    X['OverageMinutes'] = df['Total_Revolving_Bal'].astype(float)
    X['PercChangeMinutes'] = df['Total_Ct_Chng_Q4_Q1'].astype(float)
    X['PercChangeRevenues'] = df['Total_Amt_Chng_Q4_Q1'].astype(float)
    X['MonthsInService'] = df['Months_on_book'].astype(float)
    X['UniqueSubs'] = df['Total_Relationship_Count'].astype(float)
    X['ActiveSubs'] = df['Total_Relationship_Count'].astype(float)
    X['AgeHH1'] = df['Customer_Age'].astype(float)
    X['AgeHH2'] = df['Customer_Age'].astype(float)
    X['ChildrenInHH'] = np.where(df['Dependent_count'] > 0, 'Yes', 'No')
    X['RetentionCalls'] = df['Contacts_Count_12_mon'].astype(float)
    X['CustomerCareCalls'] = df['Contacts_Count_12_mon'].astype(float)
    X['MadeCallToRetentionTeam'] = np.where(df['Contacts_Count_12_mon'] > 0, 'Yes', 'No')
    X['IncomeGroup'] = map_income_category(df['Income_Category'])
    X['MaritalStatus'] = map_marital_status(df['Marital_Status'])
    X['CreditRating'] = derive_credit_rating_bank(df)
    X['HasCreditCard'] = 'Yes'
    return X


def build_portable_bank_frame(df: pd.DataFrame):
    X = pd.DataFrame(index=df.index)
    X['tenure'] = df['Months_on_book'].astype(float)
    X['age'] = df['Customer_Age'].astype(float)
    X['income_group'] = map_income_category(df['Income_Category'])
    X['marital_status'] = map_marital_status(df['Marital_Status'])
    X['children_present'] = np.where(df['Dependent_count'] > 0, 'Yes', 'No')
    X['relationship_depth'] = df['Total_Relationship_Count'].astype(float)
    X['transaction_volume'] = df['Total_Trans_Ct'].astype(float)
    X['monetary_volume'] = df['Total_Trans_Amt'].astype(float)
    X['balance_capacity'] = df['Credit_Limit'].astype(float)
    X['utilization_or_overage'] = df['Total_Revolving_Bal'].astype(float)
    X['change_volume'] = df['Total_Ct_Chng_Q4_Q1'].astype(float)
    X['change_amount'] = df['Total_Amt_Chng_Q4_Q1'].astype(float)
    X['contact_intensity'] = df['Contacts_Count_12_mon'].astype(float)
    X['credit_rating'] = derive_credit_rating_bank(df)
    X['credit_card_holder'] = 'Yes'
    return X


def build_pipeline(X: pd.DataFrame):
    numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    preprocessor = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), categorical_cols),
    ])
    model = XGBClassifier(
        **BEST_PARAMS,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return Pipeline([('preprocess', preprocessor), ('model', model)])


def compute_metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        'roc_auc': roc_auc_score(y_true, prob),
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
    }


def best_f1_threshold(y_true, prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = []
    for t in thresholds:
        scores.append(f1_score(y_true, (prob >= t).astype(int), zero_division=0))
    idx = int(np.argmax(scores))
    return float(thresholds[idx]), float(scores[idx])


def md_table(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    header = '| ' + ' | '.join(cols) + ' |
'
    header += '|' + '|'.join(['---'] * len(cols)) + '|
'
    rows = []
    for _, row in df.iterrows():
        rows.append('| ' + ' | '.join(str(row[c]) for c in cols) + ' |')
    return header + '
'.join(rows)


cell_df = load_cell2cell(ROOT / 'data' / 'raw' / 'cell2celltrain.csv')
bank_df = load_bank(ROOT / 'data' / 'external' / 'credit_card_churn' / 'BankChurners.csv')

full_X, y = build_full_cell_frame(cell_df)
portable_X, _ = build_portable_cell_frame(cell_df)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    full_X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)
X_train_port, X_test_port, _, _ = train_test_split(
    portable_X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

scale_pos_weight = int((y_train == 0).sum()) / max(int(y_train.sum()), 1)

full_pipe = build_pipeline(X_train_full)
portable_pipe = build_pipeline(X_train_port)

full_pipe.fit(X_train_full, y_train)
portable_pipe.fit(X_train_port, y_train)

full_source_prob = full_pipe.predict_proba(X_test_full)[:, 1]
portable_source_prob = portable_pipe.predict_proba(X_test_port)[:, 1]

bank_full_X = build_full_bank_frame(bank_df)
bank_port_X = build_portable_bank_frame(bank_df)
y_bank = bank_df['Attrition_Flag']

full_bank_prob = full_pipe.predict_proba(bank_full_X)[:, 1]
portable_bank_prob = portable_pipe.predict_proba(bank_port_X)[:, 1]

full_source_metrics = compute_metrics(y_test, full_source_prob)
portable_source_metrics = compute_metrics(y_test, portable_source_prob)
full_bank_metrics = compute_metrics(y_bank, full_bank_prob)
portable_bank_metrics = compute_metrics(y_bank, portable_bank_prob)

full_bank_t, full_bank_best_f1 = best_f1_threshold(y_bank, full_bank_prob)
portable_bank_t, portable_bank_best_f1 = best_f1_threshold(y_bank, portable_bank_prob)
full_bank_best_metrics = compute_metrics(y_bank, full_bank_prob, threshold=full_bank_t)
portable_bank_best_metrics = compute_metrics(y_bank, portable_bank_prob, threshold=portable_bank_t)

source_table = pd.DataFrame([
    {'model': 'Full model', **full_source_metrics},
    {'model': 'Portable model', **portable_source_metrics},
])
external_table = pd.DataFrame([
    {'model': 'Full model', **full_bank_metrics},
    {'model': 'Portable model', **portable_bank_metrics},
])
threshold_table = pd.DataFrame([
    {'model': 'Full model', 'best_threshold': full_bank_t, 'best_f1': full_bank_best_f1, **full_bank_best_metrics},
    {'model': 'Portable model', 'best_threshold': portable_bank_t, 'best_f1': portable_bank_best_f1, **portable_bank_best_metrics},
])

print('Cell2Cell holdout results (threshold=0.5)')
print(md_table(source_table.round(4)))
print('
BankChurners external results (threshold=0.5)')
print(md_table(external_table.round(4)))
print('
BankChurners threshold sensitivity')
print(md_table(threshold_table.round(4)))

result_text = []
result_text.append('# Cross-Domain Benchmark Results')
result_text.append('')
result_text.append('## Cell2Cell Holdout (threshold = 0.5)')
result_text.append(md_table(source_table.round(4)))
result_text.append('')
result_text.append('## BankChurners External Test (threshold = 0.5)')
result_text.append(md_table(external_table.round(4)))
result_text.append('')
result_text.append('## BankChurners Threshold Sensitivity')
result_text.append(md_table(threshold_table.round(4)))
result_text.append('')
result_text.append('## Portable Feature Schema')
result_text.append('- tenure')
result_text.append('- age')
result_text.append('- income_group')
result_text.append('- marital_status')
result_text.append('- children_present')
result_text.append('- relationship_depth')
result_text.append('- transaction_volume')
result_text.append('- monetary_volume')
result_text.append('- balance_capacity')
result_text.append('- utilization_or_overage')
result_text.append('- change_volume')
result_text.append('- change_amount')
result_text.append('- contact_intensity')
result_text.append('- credit_rating')
result_text.append('- credit_card_holder')
result_text.append('')
result_text.append('## Notes')
result_text.append('- Full model uses the original Cell2Cell feature space and receives many missing values on BankChurners. This is the strict external stress test.')
result_text.append('- Portable model uses a reduced schema built from common concepts that can be mapped to BankChurners.')
result_text.append('- On BankChurners, both models keep similar ranking quality, but calibration differs; threshold sensitivity is included as a reference.')

RESULT_PATH.write_text('
'.join(result_text), encoding='utf-8')
print(f'Wrote {RESULT_PATH}')


Cell2Cell holdout results (threshold=0.5)
| model | roc_auc | accuracy | precision | recall | f1 |
|---|---:|---:|---:|---:|---:|
| Full model | 0.6805 | 0.6228 | 0.4034 | 0.6451 | 0.4964 |
| Portable model | 0.6669 | 0.6190 | 0.3973 | 0.6230 | 0.4852 |

BankChurners external results (threshold=0.5)
| model | roc_auc | accuracy | precision | recall | f1 |
|---|---:|---:|---:|---:|---:|
| Full model | 0.5732 | 0.4577 | 0.1851 | 0.6982 | 0.2926 |
| Portable model | 0.5663 | 0.3044 | 0.1659 | 0.8267 | 0.2764 |

BankChurners threshold sensitivity
| model | best_threshold | best_f1 | roc_auc | accuracy | precision | recall | f1 |
|---|---:|---:|---:|---:|---:|---:|---:|
| Full model | 0.5000 | 0.2926 | 0.5732 | 0.4577 | 0.1851 | 0.6982 | 0.2926 |
| Portable model | 0.5700 | 0.2925 | 0.5663 | 0.5205 | 0.1917 | 0.6171 | 0.2925 |
Wrote /home/ms990728/소정해/results/cross_domain_benchmark_results.md


## 해석

- Full model은 Cell2Cell 내부에서는 더 좋지만, BankChurners 외부에서는 성능이 크게 떨어진다.
- Portable model은 외부에서 AUC가 크게 개선되지는 않았고, default threshold 기준 F1도 Full model보다 낮았다.
- 다만 외부 데이터에서 임계값을 다시 맞추면 두 모델의 F1이 거의 비슷해져서, ranking 자체는 큰 차이가 없고 calibration 차이가 있다는 점을 보여준다.
- 즉, 현재 portable schema는 범용성의 방향은 맞지만, 외부 업종에 대해 압도적으로 강하다고 보기는 어렵다.
